# Train/test navigator with 3 visual paths

This notebook:

1. Loads three manually collected visual-path logs (`rgb.npy`, `depth.npy`, `metadata.json`).
2. Generates the visual memory for each path using the adaptive keyframe-selection logic.
3. Processes each log using the new `update_memory` duplication rule.
4. Extracts pair-dependent features for each processed sample:
   - translation `(tx, ty, tz)`
   - rotation quaternion `(qw, qx, qy, qz)`
   - geometric registration error `rmse`
   - RGB-D visual similarity `sim_score`
5. Trains an MLP navigator with two VPs and evaluates on the held-out VP.
6. Saves the selected model and scaler.
7. Visualizes curves, metrics, confusion matrix, and some good/bad predictions.

Held-out evaluation path: `rep_rep_bed_tv-1_224_20260329_234304`.

Assumptions:
- `visual_memory_selector.py` exists in the working directory.
- `modules.rgbd_similarity` and `modules.feature_based_point_cloud_registration` are importable.
- The Habitat project paths below match your machine.

In [1]:
import os
import json
import yaml
import joblib
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)

import torch
import quaternion

from modules.rgbd_similarity import RGBDSimilarity
from modules.feature_based_point_cloud_registration import FeatureBasedPointCloudRegistration

try:
    from visual_memory_selector import VisualMemorySelector
except Exception as e:
    raise ImportError(
        "Could not import VisualMemorySelector from visual_memory_selector.py. "
        "Make sure that file is in the working directory."
    )

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

/home/rodriguez/miniconda3/envs/habitat/lib/python3.9/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/examples/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------

BASE_DIR = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/simple_logs"

# Training VPs
TRAIN_LOGS = [
    {
        "name": "rep_dinning",
        "rgb": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_metadata.json"),
    },
    {
        "name": "rep_wc",
        "rgb": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_metadata.json"),
    },
]

# Held-out evaluation VP
TEST_LOG = {
    "name": "rep_bed_tv",
    "rgb": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_rgb.npy"),
    "depth": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_depth.npy"),
    "meta": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_metadata.json"),
}

# Adaptive visual-memory selection
SIGMA_K = 2.0
SAMPLE_RATE = 1

# Feature extraction / registration config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_NAV_CONF = "LightGlue"
FEATURE_MODE = "mnn"
ID_RUN_FOR_FEATURES = 999

# Path to YAML used by FeatureBasedPointCloudRegistration
EXAMPLES_CONFIG_PATH = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/examples/config.yaml"

# Output
OUTPUT_DIR = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training
RANDOM_STATE = 42
VAL_SIZE = 0.25

print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)

Device: cuda
Output dir: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs


In [3]:
# ------------------------------------------------------------------
# HELPERS: loading, preprocessing, feature extraction
# ------------------------------------------------------------------

@dataclass
class LoadedLog:
    name: str
    rgb_frames: np.ndarray
    depth_frames: np.ndarray
    metadata: dict
    steps: List[dict]
    selected_indices: List[int]
    selection_result: dict


def load_single_log(log_cfg: Dict, selector: VisualMemorySelector) -> LoadedLog:
    rgb_frames = np.load(log_cfg["rgb"])
    depth_frames = np.load(log_cfg["depth"])
    with open(log_cfg["meta"], "r", encoding="utf-8") as f:
        metadata = json.load(f)
    steps = metadata["steps"]

    if len(rgb_frames) != len(steps):
        raise ValueError(f"Mismatch in {log_cfg['name']}: {len(rgb_frames)} rgb frames vs {len(steps)} metadata steps")

    selection_result = selector.select_from_arrays(
        rgb_frames=rgb_frames,
        depth_frames=depth_frames,
        sample_rate=SAMPLE_RATE,
    )
    selected_indices = selection_result["selected_real_indices"]

    return LoadedLog(
        name=log_cfg["name"],
        rgb_frames=rgb_frames,
        depth_frames=depth_frames,
        metadata=metadata,
        steps=steps,
        selected_indices=selected_indices,
        selection_result=selection_result,
    )


def build_segment_lookup(selected_indices: List[int], num_frames: int) -> List[Optional[int]]:
    selected_sorted = sorted(selected_indices)
    target_lookup: List[Optional[int]] = [None] * num_frames

    for k in range(len(selected_sorted) - 1):
        start_kf = selected_sorted[k]
        next_kf = selected_sorted[k + 1]
        for i in range(start_kf, next_kf):
            target_lookup[i] = next_kf

    last_kf = selected_sorted[-1]
    target_lookup[last_kf] = None
    return target_lookup


def preprocess_log_for_training(log_obj: LoadedLog) -> pd.DataFrame:
    steps = log_obj.steps
    selected_indices = log_obj.selected_indices
    num_frames = len(steps)

    selected_set = set(selected_indices)
    selected_sorted = sorted(selected_indices)
    active_target_lookup = build_segment_lookup(selected_sorted, num_frames)

    next_keyframe_map = {}
    for k in range(len(selected_sorted) - 1):
        next_keyframe_map[selected_sorted[k]] = selected_sorted[k + 1]
    next_keyframe_map[selected_sorted[-1]] = None

    rows = []

    for step in steps:
        frame_id = int(step["frame_id"])
        user_command = step["user_command"]

        if frame_id in selected_set:
            # Row 1: reached selected keyframe -> update memory
            rows.append({
                "source_log": log_obj.name,
                "raw_frame_id": frame_id,
                "user_command": user_command,
                "processed_command": "update_memory",
                "target_keyframe": frame_id,
                "row_type": "selected_frame_update",
                "position": step.get("position", None),
                "rotation_quaternion_wxyz": step.get("rotation_quaternion_wxyz", None),
            })

            # Row 2: duplicated row with next target if possible
            next_kf = next_keyframe_map[frame_id]
            if next_kf is not None and user_command != "finish":
                rows.append({
                    "source_log": log_obj.name,
                    "raw_frame_id": frame_id,
                    "user_command": user_command,
                    "processed_command": user_command,
                    "target_keyframe": next_kf,
                    "row_type": "selected_frame_duplicate_next_target",
                    "position": step.get("position", None),
                    "rotation_quaternion_wxyz": step.get("rotation_quaternion_wxyz", None),
                })
        else:
            target_kf = active_target_lookup[frame_id]
            rows.append({
                "source_log": log_obj.name,
                "raw_frame_id": frame_id,
                "user_command": user_command,
                "processed_command": user_command,
                "target_keyframe": target_kf,
                "row_type": "regular_motion",
                "position": step.get("position", None),
                "rotation_quaternion_wxyz": step.get("rotation_quaternion_wxyz", None),
            })

    df = pd.DataFrame(rows)
    return df


def normalize_label(label: str) -> str:
    if label == "update_memory":
        return "update"
    return label


def quaternion_to_wxyz(q_obj) -> Tuple[float, float, float, float]:
    if hasattr(q_obj, "w") and hasattr(q_obj, "x") and hasattr(q_obj, "y") and hasattr(q_obj, "z"):
        return float(q_obj.w), float(q_obj.x), float(q_obj.y), float(q_obj.z)
    arr = quaternion.as_float_array(q_obj)
    return float(arr[0]), float(arr[1]), float(arr[2]), float(arr[3])


def load_feature_registration_config() -> dict:
    with open(EXAMPLES_CONFIG_PATH, "r") as f:
        return yaml.safe_load(f)


def build_feature_extractors(device: str):
    config = load_feature_registration_config()

    rgbd_similarity = RGBDSimilarity(device=device)

    feature_registration = FeatureBasedPointCloudRegistration(
        config=config,
        device=device,
        id_run=ID_RUN_FOR_FEATURES,
        feature_nav_conf=FEATURE_NAV_CONF,
        feature_mode=FEATURE_MODE,
        topological_map=None,
        manual_operation=False,
    )

    return rgbd_similarity, feature_registration


def extract_features_from_processed_df(
    processed_df: pd.DataFrame,
    loaded_logs: Dict[str, LoadedLog],
    rgbd_similarity: RGBDSimilarity,
    feature_registration: FeatureBasedPointCloudRegistration,
) -> pd.DataFrame:
    rows_out = []

    iterable = processed_df.to_dict("records")
    for row in tqdm(iterable, total=len(iterable), desc="Extracting pair-dependent features"):
        label = normalize_label(row["processed_command"])
        target_kf = row["target_keyframe"]

        # Skip terminal rows or rows without a valid target keyframe
        if label == "finish":
            continue
        if target_kf is None or (isinstance(target_kf, float) and np.isnan(target_kf)):
            continue

        log_obj = loaded_logs[row["source_log"]]
        raw_frame_id = int(row["raw_frame_id"])
        target_kf = int(target_kf)

        o_color = log_obj.rgb_frames[raw_frame_id]
        o_depth = log_obj.depth_frames[raw_frame_id]
        k_color = log_obj.rgb_frames[target_kf]
        k_depth = log_obj.depth_frames[target_kf]

        try:
            sim_score = float(rgbd_similarity.compute_image_similarity(o_color, o_depth, k_color, k_depth))

            bot_lost, est_quaternion, rmse, est_t_source_target, _ = feature_registration.compute_relative_pose(
                o_color,
                o_depth,
                k_color,
                k_depth,
            )

            qw, qx, qy, qz = quaternion_to_wxyz(est_quaternion)
            tx, ty, tz = [float(x) for x in np.asarray(est_t_source_target).reshape(-1)[:3]]
            rmse = float(rmse)

            rows_out.append({
                "source_log": row["source_log"],
                "raw_frame_id": raw_frame_id,
                "target_keyframe": target_kf,
                "user_command": row["user_command"],
                "processed_command": row["processed_command"],
                "label": label,
                "row_type": row["row_type"],
                "bot_lost": bool(bot_lost),
                "rmse": rmse,
                "tx": tx,
                "ty": ty,
                "tz": tz,
                "qw": qw,
                "qx": qx,
                "qy": qy,
                "qz": qz,
                "sim_score": sim_score,
            })
        except Exception as e:
            # Skip problematic rows, but keep a visible trace
            rows_out.append({
                "source_log": row["source_log"],
                "raw_frame_id": raw_frame_id,
                "target_keyframe": target_kf,
                "user_command": row["user_command"],
                "processed_command": row["processed_command"],
                "label": label,
                "row_type": row["row_type"],
                "bot_lost": None,
                "rmse": np.nan,
                "tx": np.nan,
                "ty": np.nan,
                "tz": np.nan,
                "qw": np.nan,
                "qx": np.nan,
                "qy": np.nan,
                "qz": np.nan,
                "sim_score": np.nan,
                "error": str(e),
            })

    feat_df = pd.DataFrame(rows_out)
    return feat_df

In [4]:
# ------------------------------------------------------------------
# LOAD LOGS AND GENERATE VISUAL MEMORIES
# ------------------------------------------------------------------

selector = VisualMemorySelector(sigma_k=SIGMA_K)

train_loaded_logs = {}
for log_cfg in TRAIN_LOGS:
    loaded = load_single_log(log_cfg, selector)
    train_loaded_logs[loaded.name] = loaded
    print(f"Loaded train log: {loaded.name}")
    print(f"  frames: {len(loaded.rgb_frames)}")
    print(f"  selected keyframes: {loaded.selected_indices}")
    print(f"  threshold: {loaded.selection_result['threshold']:.4f}")

test_loaded = load_single_log(TEST_LOG, selector)
print(f"Loaded test log: {test_loaded.name}")
print(f"  frames: {len(test_loaded.rgb_frames)}")
print(f"  selected keyframes: {test_loaded.selected_indices}")
print(f"  threshold: {test_loaded.selection_result['threshold']:.4f}")

Computing similarity matrix: 100%|██████████| 266/266 [00:08<00:00, 32.25it/s]


Loaded train log: rep_dinning
  frames: 266
  selected keyframes: [0, 3, 5, 8, 12, 17, 21, 24, 28, 30, 32, 35, 38, 40, 43, 45, 47, 49, 51, 53, 55, 58, 61, 62, 66, 69, 72, 74, 77, 80, 82, 86, 91, 97, 99, 103, 106, 109, 113, 116, 118, 120, 123, 124, 127, 130, 134, 136, 139, 141, 143, 145, 147, 149, 151, 153, 157, 160, 163, 165, 169, 172, 175, 178, 181, 183, 187, 189, 191, 193, 195, 197, 199, 201, 204, 207, 209, 211, 214, 217, 219, 222, 225, 228, 230, 232, 234, 236, 238, 240, 243, 246, 247, 248, 250, 254, 256, 258, 259, 262, 265]
  threshold: 0.9280


Computing similarity matrix: 100%|██████████| 298/298 [00:10<00:00, 29.02it/s]


Loaded train log: rep_wc
  frames: 298
  selected keyframes: [0, 5, 8, 9, 11, 13, 15, 17, 20, 21, 22, 24, 25, 26, 28, 29, 30, 33, 37, 40, 44, 47, 49, 51, 54, 56, 59, 62, 63, 67, 69, 72, 74, 77, 79, 81, 83, 86, 88, 91, 93, 96, 98, 100, 103, 107, 110, 112, 115, 118, 120, 123, 125, 128, 131, 134, 137, 139, 141, 144, 147, 151, 155, 157, 161, 165, 168, 170, 173, 176, 178, 181, 182, 185, 188, 190, 193, 195, 197, 200, 202, 204, 207, 209, 210, 212, 215, 218, 223, 230, 235, 239, 243, 246, 249, 252, 256, 259, 263, 266, 270, 273, 277, 280, 282, 285, 288, 291, 294, 297]
  threshold: 0.9193


Computing similarity matrix: 100%|██████████| 225/225 [00:05<00:00, 38.56it/s]

Loaded test log: rep_bed_tv
  frames: 225
  selected keyframes: [0, 3, 5, 7, 9, 11, 13, 16, 19, 21, 23, 26, 28, 30, 32, 34, 36, 38, 42, 45, 48, 50, 52, 55, 58, 61, 63, 65, 67, 68, 70, 72, 73, 75, 79, 81, 82, 85, 87, 88, 89, 92, 96, 101, 103, 105, 110, 114, 116, 118, 121, 123, 125, 127, 129, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 161, 165, 169, 173, 178, 182, 187, 191, 196, 200, 204, 208, 212, 215, 219, 223, 224]
  threshold: 0.9241


In [5]:
# ------------------------------------------------------------------
# VISUAL-MEMORY SUMMARY TABLE
# ------------------------------------------------------------------

summary_rows = []
for name, log_obj in train_loaded_logs.items():
    summary_rows.append({
        "log": name,
        "num_frames": len(log_obj.rgb_frames),
        "num_keyframes": len(log_obj.selected_indices),
        "threshold": log_obj.selection_result["threshold"],
        "mu_d1": log_obj.selection_result["mu"],
        "sigma_d1": log_obj.selection_result["sigma"],
    })
summary_rows.append({
    "log": test_loaded.name,
    "num_frames": len(test_loaded.rgb_frames),
    "num_keyframes": len(test_loaded.selected_indices),
    "threshold": test_loaded.selection_result["threshold"],
    "mu_d1": test_loaded.selection_result["mu"],
    "sigma_d1": test_loaded.selection_result["sigma"],
})

selection_summary_df = pd.DataFrame(summary_rows)
display(selection_summary_df)

,log,num_frames,num_keyframes,threshold,mu_d1,sigma_d1
0,rep_dinning,266,101,0.928036,0.951678,0.011821
1,rep_wc,298,110,0.919279,0.949984,0.015353
2,rep_bed_tv,225,87,0.924090,0.952792,0.014351


In [6]:
# ------------------------------------------------------------------
# PREPROCESS LOGS WITH NEW UPDATE-MEMORY RULE
# ------------------------------------------------------------------

processed_train_dfs = []
for name, log_obj in train_loaded_logs.items():
    df = preprocess_log_for_training(log_obj)
    processed_train_dfs.append(df)
    print(f"Processed train log {name}: shape={df.shape}")

processed_train_df = pd.concat(processed_train_dfs, ignore_index=True)
processed_test_df = preprocess_log_for_training(test_loaded)

print("Combined train processed shape:", processed_train_df.shape)
print("Held-out test processed shape:", processed_test_df.shape)

display(processed_train_df.head(20))

Processed train log rep_dinning: shape=(366, 8)
Processed train log rep_wc: shape=(407, 8)
Combined train processed shape: (773, 8)
Held-out test processed shape: (311, 8)


,source_log,raw_frame_id,user_command,processed_command,target_keyframe,row_type,position,rotation_quaternion_wxyz
0,rep_dinning,0,forward,update_memory,0,selected_frame_update,"[0.6751305460929871, 0.07244700193405151, -3.2...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
1,rep_dinning,0,forward,forward,3,selected_frame_duplicate_next_target,"[0.6751305460929871, 0.07244700193405151, -3.2...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
2,rep_dinning,1,forward,forward,3,regular_motion,"[0.6560497879981995, 0.07244700193405151, -3.1...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
3,rep_dinning,2,forward,forward,3,regular_motion,"[0.6369690299034119, 0.07244700193405151, -3.0...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
4,rep_dinning,3,forward,update_memory,3,selected_frame_update,"[0.6178882718086243, 0.07244700193405151, -2.9...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
5,rep_dinning,3,forward,forward,5,selected_frame_duplicate_next_target,"[0.6178882718086243, 0.07244700193405151, -2.9...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
6,rep_dinning,4,forward,forward,5,regular_motion,"[0.5988075137138367, 0.07244700193405151, -2.8...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
7,rep_dinning,5,forward,update_memory,5,selected_frame_update,"[0.5797267556190491, 0.07244700193405151, -2.7...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
8,rep_dinning,5,forward,forward,8,selected_frame_duplicate_next_target,"[0.5797267556190491, 0.07244700193405151, -2.7...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
9,rep_dinning,6,forward,forward,8,regular_motion,"[0.5606459975242615, 0.07244700193405151, -2.6...","[0.09584498405456543, 0.0, 0.9953963160514832,..."


In [7]:
# ------------------------------------------------------------------
# PREPROCESSED LABEL COUNTS (BEFORE FEATURE EXTRACTION)
# ------------------------------------------------------------------

print("Train processed command counts:")
display(processed_train_df["processed_command"].value_counts(dropna=False).to_frame("count"))

print("Test processed command counts:")
display(processed_test_df["processed_command"].value_counts(dropna=False).to_frame("count"))

Train processed command counts:


,count
processed_command,
forward,301
update_memory,211
right,151
left,110


Test processed command counts:


,count
processed_command,
forward,127
update_memory,87
right,51
left,46


In [8]:
# ------------------------------------------------------------------
# BUILD FEATURE EXTRACTORS
# ------------------------------------------------------------------

rgbd_similarity, feature_registration = build_feature_extractors(DEVICE)
print("Feature extractors ready.")

Feature extractors ready.


In [9]:
# ------------------------------------------------------------------
# EXTRACT FEATURES FOR TRAINING SET
# ------------------------------------------------------------------

all_train_logs = dict(train_loaded_logs)

train_feat_df = extract_features_from_processed_df(
    processed_df=processed_train_df,
    loaded_logs=all_train_logs,
    rgbd_similarity=rgbd_similarity,
    feature_registration=feature_registration,
)

print("Train feature table shape:", train_feat_df.shape)
print(train_feat_df.shape)
display(train_feat_df.head(20))

Extracting pair-dependent features: 100%|██████████| 773/773 [02:18<00:00,  5.58it/s]

Train feature table shape: (773, 17)
(773, 17)


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_dinning,0,0,forward,update_memory,update,selected_frame_update,False,7.971944e-16,-1.110223e-16,7.771561e-16,-8.881784e-16,-1.000000,7.793685e-17,-2.010076e-16,-9.748082e-17,1.000000
1,rep_dinning,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,3.522270e-01,-9.383411e-02,4.765055e-02,2.213170e-01,-0.999881,6.338716e-03,1.401337e-02,1.289226e-03,0.910583
2,rep_dinning,1,3,forward,forward,forward,regular_motion,False,2.861813e-01,-6.898344e-03,-1.306482e-02,1.878601e-01,-0.999999,-6.712459e-04,1.242131e-03,-7.703627e-04,0.934426
3,rep_dinning,2,3,forward,forward,forward,regular_motion,False,5.026970e-01,5.892855e-02,-1.529649e-02,8.695854e-02,0.999978,2.039822e-03,6.120563e-03,-1.279138e-03,0.955410
4,rep_dinning,3,3,forward,update_memory,update,selected_frame_update,False,5.018928e-16,3.330669e-16,1.942890e-16,-4.440892e-16,1.000000,-1.387779e-17,6.839902e-17,2.358004e-17,1.000000
5,rep_dinning,3,5,forward,forward,forward,selected_frame_duplicate_next_target,False,2.615858e-01,-5.096559e-02,-5.202617e-03,1.878503e-01,-0.999985,4.365208e-05,5.272024e-03,-1.423161e-03,0.921255
6,rep_dinning,4,5,forward,forward,forward,regular_motion,False,3.717926e-01,-5.785748e-03,6.101854e-03,1.024900e-01,-0.999999,9.205683e-04,9.102691e-04,6.236159e-04,0.947549
7,rep_dinning,5,5,forward,update_memory,update,selected_frame_update,False,1.467330e-15,-5.551115e-16,-1.665335e-15,1.776357e-15,-1.000000,-1.505788e-16,-2.568621e-16,4.175193e-17,1.000000
8,rep_dinning,5,8,forward,forward,forward,selected_frame_duplicate_next_target,False,2.922420e-01,-4.770651e-02,-4.390749e-02,2.646471e-01,-0.999958,-4.840289e-03,7.688849e-03,-7.163035e-04,0.901263
9,rep_dinning,6,8,forward,forward,forward,regular_motion,False,2.272391e-01,1.425079e-04,1.322961e-02,1.883300e-01,-0.999997,2.264237e-03,8.407662e-04,-4.265545e-04,0.948340


In [11]:
print("Raw extracted train feature shape:", train_feat_df.shape)
display(train_feat_df.head(20))
cols_to_check = ["rmse","tx","ty","tz","qw","qx","qy","qz","sim_score"]
if "error" in train_feat_df.columns:
    cols_to_check.append("error")

print(train_feat_df[cols_to_check].isna().sum())

if "error" in train_feat_df.columns:
    display(train_feat_df["error"].fillna("NO_ERROR").value_counts().head(20))

Raw extracted train feature shape: (773, 17)


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_dinning,0,0,forward,update_memory,update,selected_frame_update,False,7.971944e-16,-1.110223e-16,7.771561e-16,-8.881784e-16,-1.000000,7.793685e-17,-2.010076e-16,-9.748082e-17,1.000000
1,rep_dinning,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,3.522270e-01,-9.383411e-02,4.765055e-02,2.213170e-01,-0.999881,6.338716e-03,1.401337e-02,1.289226e-03,0.910583
2,rep_dinning,1,3,forward,forward,forward,regular_motion,False,2.861813e-01,-6.898344e-03,-1.306482e-02,1.878601e-01,-0.999999,-6.712459e-04,1.242131e-03,-7.703627e-04,0.934426
3,rep_dinning,2,3,forward,forward,forward,regular_motion,False,5.026970e-01,5.892855e-02,-1.529649e-02,8.695854e-02,0.999978,2.039822e-03,6.120563e-03,-1.279138e-03,0.955410
4,rep_dinning,3,3,forward,update_memory,update,selected_frame_update,False,5.018928e-16,3.330669e-16,1.942890e-16,-4.440892e-16,1.000000,-1.387779e-17,6.839902e-17,2.358004e-17,1.000000
5,rep_dinning,3,5,forward,forward,forward,selected_frame_duplicate_next_target,False,2.615858e-01,-5.096559e-02,-5.202617e-03,1.878503e-01,-0.999985,4.365208e-05,5.272024e-03,-1.423161e-03,0.921255
6,rep_dinning,4,5,forward,forward,forward,regular_motion,False,3.717926e-01,-5.785748e-03,6.101854e-03,1.024900e-01,-0.999999,9.205683e-04,9.102691e-04,6.236159e-04,0.947549
7,rep_dinning,5,5,forward,update_memory,update,selected_frame_update,False,1.467330e-15,-5.551115e-16,-1.665335e-15,1.776357e-15,-1.000000,-1.505788e-16,-2.568621e-16,4.175193e-17,1.000000
8,rep_dinning,5,8,forward,forward,forward,selected_frame_duplicate_next_target,False,2.922420e-01,-4.770651e-02,-4.390749e-02,2.646471e-01,-0.999958,-4.840289e-03,7.688849e-03,-7.163035e-04,0.901263
9,rep_dinning,6,8,forward,forward,forward,regular_motion,False,2.272391e-01,1.425079e-04,1.322961e-02,1.883300e-01,-0.999997,2.264237e-03,8.407662e-04,-4.265545e-04,0.948340


rmse         0
tx           0
ty           0
tz           0
qw           0
qx           0
qy           0
qz           0
sim_score    0
dtype: int64


In [12]:
print(processed_train_df.shape)
display(processed_train_df.head(20))
print(processed_train_df["processed_command"].value_counts(dropna=False))
print(processed_train_df["target_keyframe"].isna().sum(), "rows with target_keyframe NaN")
display(processed_train_df[processed_train_df["target_keyframe"].isna()].head(20))

(773, 8)


,source_log,raw_frame_id,user_command,processed_command,target_keyframe,row_type,position,rotation_quaternion_wxyz
0,rep_dinning,0,forward,update_memory,0,selected_frame_update,"[0.6751305460929871, 0.07244700193405151, -3.2...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
1,rep_dinning,0,forward,forward,3,selected_frame_duplicate_next_target,"[0.6751305460929871, 0.07244700193405151, -3.2...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
2,rep_dinning,1,forward,forward,3,regular_motion,"[0.6560497879981995, 0.07244700193405151, -3.1...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
3,rep_dinning,2,forward,forward,3,regular_motion,"[0.6369690299034119, 0.07244700193405151, -3.0...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
4,rep_dinning,3,forward,update_memory,3,selected_frame_update,"[0.6178882718086243, 0.07244700193405151, -2.9...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
5,rep_dinning,3,forward,forward,5,selected_frame_duplicate_next_target,"[0.6178882718086243, 0.07244700193405151, -2.9...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
6,rep_dinning,4,forward,forward,5,regular_motion,"[0.5988075137138367, 0.07244700193405151, -2.8...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
7,rep_dinning,5,forward,update_memory,5,selected_frame_update,"[0.5797267556190491, 0.07244700193405151, -2.7...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
8,rep_dinning,5,forward,forward,8,selected_frame_duplicate_next_target,"[0.5797267556190491, 0.07244700193405151, -2.7...","[0.09584498405456543, 0.0, 0.9953963160514832,..."
9,rep_dinning,6,forward,forward,8,regular_motion,"[0.5606459975242615, 0.07244700193405151, -2.6...","[0.09584498405456543, 0.0, 0.9953963160514832,..."


processed_command
forward          301
update_memory    211
right            151
left             110
Name: count, dtype: int64
0 rows with target_keyframe NaN


,source_log,raw_frame_id,user_command,processed_command,target_keyframe,row_type,position,rotation_quaternion_wxyz


In [13]:
display(processed_train_df[["raw_frame_id","processed_command","target_keyframe","row_type"]].tail(30))

,raw_frame_id,processed_command,target_keyframe,row_type
743,275,forward,277,regular_motion
744,276,forward,277,regular_motion
745,277,update_memory,277,selected_frame_update
746,277,forward,280,selected_frame_duplicate_next_target
747,278,forward,280,regular_motion
748,279,forward,280,regular_motion
749,280,update_memory,280,selected_frame_update
750,280,forward,282,selected_frame_duplicate_next_target
751,281,forward,282,regular_motion
752,282,update_memory,282,selected_frame_update


In [14]:
print("processed_train_df shape:", processed_train_df.shape)
print("processed_test_df shape :", processed_test_df.shape)

processed_train_df shape: (773, 8)
processed_test_df shape : (311, 8)


In [15]:
# ------------------------------------------------------------------
# EXTRACT FEATURES FOR HELD-OUT TEST SET
# ------------------------------------------------------------------

all_test_logs = {test_loaded.name: test_loaded}

test_feat_df = extract_features_from_processed_df(
    processed_df=processed_test_df,
    loaded_logs=all_test_logs,
    rgbd_similarity=rgbd_similarity,
    feature_registration=feature_registration,
)

print("Test feature table shape:", test_feat_df.shape)
display(test_feat_df.head(20))

Extracting pair-dependent features: 100%|██████████| 311/311 [00:55<00:00,  5.59it/s]

Test feature table shape: (311, 17)


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_bed_tv,0,0,forward,update_memory,update,selected_frame_update,False,7.586020e-16,-1.110223e-16,-8.326673e-17,-4.440892e-16,1.000000,2.974163e-17,3.670273e-17,3.502945e-16,1.000000
1,rep_bed_tv,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,5.682374e-01,-2.588980e-02,-2.374301e-02,2.532331e-01,0.999910,3.370717e-03,1.297198e-02,1.129306e-03,0.904431
2,rep_bed_tv,1,3,right,right,right,regular_motion,False,4.692756e-01,-3.569933e-02,-2.295490e-02,1.131307e-01,0.999919,3.660933e-03,1.219970e-02,5.302020e-04,0.930474
3,rep_bed_tv,2,3,forward,forward,forward,regular_motion,False,5.192502e-01,-6.531503e-02,-1.854075e-02,1.032665e-01,-0.999954,-2.414029e-03,9.194846e-03,-8.861523e-04,0.946828
4,rep_bed_tv,3,3,right,update_memory,update,selected_frame_update,False,1.078586e-15,0.000000e+00,3.053113e-16,-1.332268e-15,1.000000,-1.237869e-17,1.359306e-17,-2.544303e-16,1.000000
5,rep_bed_tv,3,5,right,right,right,selected_frame_duplicate_next_target,False,3.097040e-01,1.989162e-02,-1.775926e-02,7.119942e-02,0.999740,2.373546e-03,2.268937e-02,-6.325140e-04,0.908575
6,rep_bed_tv,4,5,forward,forward,forward,regular_motion,False,3.618185e-01,1.316743e-03,5.482810e-03,7.532176e-02,0.999997,-1.162376e-03,1.912958e-03,-8.819635e-04,0.941193
7,rep_bed_tv,5,5,right,update_memory,update,selected_frame_update,False,1.076226e-15,-1.110223e-16,-3.053113e-16,-4.440892e-16,-1.000000,5.628806e-17,1.411150e-17,3.401391e-16,1.000000
8,rep_bed_tv,5,7,right,right,right,selected_frame_duplicate_next_target,False,3.532445e-01,3.948934e-02,-1.101012e-02,1.401825e-01,0.999656,1.509478e-03,2.614926e-02,1.182444e-03,0.920854
9,rep_bed_tv,6,7,forward,forward,forward,regular_motion,False,5.228334e-01,-3.343282e-03,-3.470554e-03,7.827487e-02,0.999996,-8.494755e-05,2.655613e-03,-1.089571e-03,0.929403


In [16]:
print(train_feat_df.shape)
print(train_feat_df.columns.tolist())
display(train_feat_df.head(10))
if "error" in train_feat_df.columns:
    display(train_feat_df["error"].value_counts(dropna=False).head(20))

(773, 17)
['source_log', 'raw_frame_id', 'target_keyframe', 'user_command', 'processed_command', 'label', 'row_type', 'bot_lost', 'rmse', 'tx', 'ty', 'tz', 'qw', 'qx', 'qy', 'qz', 'sim_score']


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_dinning,0,0,forward,update_memory,update,selected_frame_update,False,7.971944e-16,-1.110223e-16,7.771561e-16,-8.881784e-16,-1.000000,7.793685e-17,-2.010076e-16,-9.748082e-17,1.000000
1,rep_dinning,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,3.522270e-01,-9.383411e-02,4.765055e-02,2.213170e-01,-0.999881,6.338716e-03,1.401337e-02,1.289226e-03,0.910583
2,rep_dinning,1,3,forward,forward,forward,regular_motion,False,2.861813e-01,-6.898344e-03,-1.306482e-02,1.878601e-01,-0.999999,-6.712459e-04,1.242131e-03,-7.703627e-04,0.934426
3,rep_dinning,2,3,forward,forward,forward,regular_motion,False,5.026970e-01,5.892855e-02,-1.529649e-02,8.695854e-02,0.999978,2.039822e-03,6.120563e-03,-1.279138e-03,0.955410
4,rep_dinning,3,3,forward,update_memory,update,selected_frame_update,False,5.018928e-16,3.330669e-16,1.942890e-16,-4.440892e-16,1.000000,-1.387779e-17,6.839902e-17,2.358004e-17,1.000000
5,rep_dinning,3,5,forward,forward,forward,selected_frame_duplicate_next_target,False,2.615858e-01,-5.096559e-02,-5.202617e-03,1.878503e-01,-0.999985,4.365208e-05,5.272024e-03,-1.423161e-03,0.921255
6,rep_dinning,4,5,forward,forward,forward,regular_motion,False,3.717926e-01,-5.785748e-03,6.101854e-03,1.024900e-01,-0.999999,9.205683e-04,9.102691e-04,6.236159e-04,0.947549
7,rep_dinning,5,5,forward,update_memory,update,selected_frame_update,False,1.467330e-15,-5.551115e-16,-1.665335e-15,1.776357e-15,-1.000000,-1.505788e-16,-2.568621e-16,4.175193e-17,1.000000
8,rep_dinning,5,8,forward,forward,forward,selected_frame_duplicate_next_target,False,2.922420e-01,-4.770651e-02,-4.390749e-02,2.646471e-01,-0.999958,-4.840289e-03,7.688849e-03,-7.163035e-04,0.901263
9,rep_dinning,6,8,forward,forward,forward,regular_motion,False,2.272391e-01,1.425079e-04,1.322961e-02,1.883300e-01,-0.999997,2.264237e-03,8.407662e-04,-4.265545e-04,0.948340


In [17]:
# ------------------------------------------------------------------
# CLEAN FEATURE TABLES AND SAVE THEM
# ------------------------------------------------------------------

feature_cols = ["rmse", "tx", "ty", "tz", "qw", "qx", "qy", "qz", "sim_score"]

train_feat_df = train_feat_df.dropna(subset=feature_cols + ["label"]).reset_index(drop=True)
test_feat_df = test_feat_df.dropna(subset=feature_cols + ["label"]).reset_index(drop=True)

train_feat_csv = os.path.join(OUTPUT_DIR, "train_features.csv")
test_feat_csv = os.path.join(OUTPUT_DIR, "test_features.csv")
train_feat_df.to_csv(train_feat_csv, index=False)
test_feat_df.to_csv(test_feat_csv, index=False)

print("Clean train feature shape:", train_feat_df.shape)
print("Clean test feature shape:", test_feat_df.shape)
print("Saved:")
print(train_feat_csv)
print(test_feat_csv)

Clean train feature shape: (773, 17)
Clean test feature shape: (311, 17)
Saved:
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/train_features.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/test_features.csv


In [18]:
# ------------------------------------------------------------------
# FEATURE-TABLE LABEL COUNTS
# ------------------------------------------------------------------

print("Train label counts:")
display(train_feat_df["label"].value_counts().to_frame("count"))

print("Test label counts:")
display(test_feat_df["label"].value_counts().to_frame("count"))

Train label counts:


,count
label,
forward,301
update,211
right,151
left,110


Test label counts:


,count
label,
forward,127
update,87
right,51
left,46


In [19]:
print("train_feat_df shape:", train_feat_df.shape)
print("test_feat_df shape :", test_feat_df.shape)
display(train_feat_df.head())
display(test_feat_df.head())

train_feat_df shape: (773, 17)
test_feat_df shape : (311, 17)


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_dinning,0,0,forward,update_memory,update,selected_frame_update,False,7.971944e-16,-1.110223e-16,7.771561e-16,-8.881784e-16,-1.000000,7.793685e-17,-2.010076e-16,-9.748082e-17,1.000000
1,rep_dinning,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,3.522270e-01,-9.383411e-02,4.765055e-02,2.213170e-01,-0.999881,6.338716e-03,1.401337e-02,1.289226e-03,0.910583
2,rep_dinning,1,3,forward,forward,forward,regular_motion,False,2.861813e-01,-6.898344e-03,-1.306482e-02,1.878601e-01,-0.999999,-6.712459e-04,1.242131e-03,-7.703627e-04,0.934426
3,rep_dinning,2,3,forward,forward,forward,regular_motion,False,5.026970e-01,5.892855e-02,-1.529649e-02,8.695854e-02,0.999978,2.039822e-03,6.120563e-03,-1.279138e-03,0.955410
4,rep_dinning,3,3,forward,update_memory,update,selected_frame_update,False,5.018928e-16,3.330669e-16,1.942890e-16,-4.440892e-16,1.000000,-1.387779e-17,6.839902e-17,2.358004e-17,1.000000


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score
0,rep_bed_tv,0,0,forward,update_memory,update,selected_frame_update,False,7.586020e-16,-1.110223e-16,-8.326673e-17,-4.440892e-16,1.000000,2.974163e-17,3.670273e-17,3.502945e-16,1.000000
1,rep_bed_tv,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,5.682374e-01,-2.588980e-02,-2.374301e-02,2.532331e-01,0.999910,3.370717e-03,1.297198e-02,1.129306e-03,0.904431
2,rep_bed_tv,1,3,right,right,right,regular_motion,False,4.692756e-01,-3.569933e-02,-2.295490e-02,1.131307e-01,0.999919,3.660933e-03,1.219970e-02,5.302020e-04,0.930474
3,rep_bed_tv,2,3,forward,forward,forward,regular_motion,False,5.192502e-01,-6.531503e-02,-1.854075e-02,1.032665e-01,-0.999954,-2.414029e-03,9.194846e-03,-8.861523e-04,0.946828
4,rep_bed_tv,3,3,right,update_memory,update,selected_frame_update,False,1.078586e-15,0.000000e+00,3.053113e-16,-1.332268e-15,1.000000,-1.237869e-17,1.359306e-17,-2.544303e-16,1.000000


In [20]:
# ------------------------------------------------------------------
# TRAIN/VAL SPLIT (ONLY FROM THE TWO TRAINING VPs)
# ------------------------------------------------------------------

X_all = train_feat_df[feature_cols].values.astype(np.float32)
y_all = train_feat_df["label"].values

X_test = test_feat_df[feature_cols].values.astype(np.float32)
y_test = test_feat_df["label"].values

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_all,
    y_all,
    np.arange(len(train_feat_df)),
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_all,
)

print("Train size:", X_train.shape[0])
print("Val size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 579
Val size: 194
Test size: 311


In [21]:
# ------------------------------------------------------------------
# TRAIN MLP CANDIDATES
# ------------------------------------------------------------------

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
X_all_s = scaler.transform(X_all)

candidate_configs = [
    {"name": "relu_adam", "activation": "relu", "solver": "adam"},
    {"name": "tanh_adam", "activation": "tanh", "solver": "adam"},
    {"name": "relu_sgd", "activation": "relu", "solver": "sgd"},
    {"name": "tanh_sgd", "activation": "tanh", "solver": "sgd"},
]

model_results = []
trained_models = {}

for cfg in candidate_configs:
    clf = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation=cfg["activation"],
        solver=cfg["solver"],
        max_iter=600,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
    )

    clf.fit(X_train_s, y_train)
    trained_models[cfg["name"]] = clf

    val_pred = clf.predict(X_val_s)
    val_bal_acc = balanced_accuracy_score(y_val, val_pred)
    val_macro_f1 = f1_score(y_val, val_pred, average="macro")
    val_weighted_f1 = f1_score(y_val, val_pred, average="weighted")
    val_acc = accuracy_score(y_val, val_pred)

    model_results.append({
        "name": cfg["name"],
        "activation": cfg["activation"],
        "solver": cfg["solver"],
        "n_iter": clf.n_iter_,
        "val_accuracy": val_acc,
        "val_balanced_accuracy": val_bal_acc,
        "val_macro_f1": val_macro_f1,
        "val_weighted_f1": val_weighted_f1,
        "final_loss": clf.loss_,
    })

results_df = pd.DataFrame(model_results).sort_values(["val_macro_f1", "val_balanced_accuracy"], ascending=False).reset_index(drop=True)
display(results_df)

,name,activation,solver,n_iter,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,final_loss
0,relu_adam,relu,adam,66,0.938144,0.941422,0.934436,0.938005,0.189602
1,tanh_adam,tanh,adam,51,0.917526,0.921598,0.915952,0.916810,0.231499
2,tanh_sgd,tanh,sgd,98,0.886598,0.896003,0.886773,0.884967,0.334332
3,relu_sgd,relu,sgd,75,0.865979,0.828170,0.846586,0.862038,0.518192


In [22]:
# ------------------------------------------------------------------
# SELECT BEST MODEL BASED ON EXTERNAL VALIDATION
# ------------------------------------------------------------------

best_model_name = results_df.iloc[0]["name"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)
print(results_df.iloc[0])

Best model: relu_adam
name                     relu_adam
activation                    relu
solver                        adam
n_iter                          66
val_accuracy              0.938144
val_balanced_accuracy     0.941422
val_macro_f1              0.934436
val_weighted_f1           0.938005
final_loss                0.189602
Name: 0, dtype: object


In [23]:
# ------------------------------------------------------------------
# SAVE MODEL AND SCALER
# ------------------------------------------------------------------

model_path = os.path.join(OUTPUT_DIR, f"navigator_{best_model_name}.joblib")
scaler_path = os.path.join(OUTPUT_DIR, f"scaler_{best_model_name}.joblib")
results_path = os.path.join(OUTPUT_DIR, "model_selection_results.csv")
config_path = os.path.join(OUTPUT_DIR, "training_config.json")

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
results_df.to_csv(results_path, index=False)

training_config = {
    "train_logs": TRAIN_LOGS,
    "test_log": TEST_LOG,
    "sigma_k": SIGMA_K,
    "sample_rate": SAMPLE_RATE,
    "feature_cols": feature_cols,
    "best_model_name": best_model_name,
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

print("Saved:")
print(model_path)
print(scaler_path)
print(results_path)
print(config_path)

Saved:
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/navigator_relu_adam.joblib
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/scaler_relu_adam.joblib
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/model_selection_results.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/training_config.json


In [24]:
# ------------------------------------------------------------------
# TRAIN / VALIDATION CURVES OF BEST MODEL
# ------------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(best_model.loss_curve_, marker="o")
plt.title(f"Loss curve - {best_model_name}")
plt.xlabel("Iteration")
plt.ylabel("Training loss")
plt.grid(True)

plt.subplot(1, 2, 2)
if hasattr(best_model, "validation_scores_") and best_model.validation_scores_ is not None:
    plt.plot(best_model.validation_scores_, marker="o")
    plt.title(f"Internal validation score curve - {best_model_name}")
    plt.xlabel("Iteration")
    plt.ylabel("Validation score")
    plt.grid(True)
else:
    plt.text(0.5, 0.5, "No validation_scores_ available", ha="center", va="center")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [25]:
# ------------------------------------------------------------------
# EXTERNAL VALIDATION PERFORMANCE
# ------------------------------------------------------------------

val_pred = best_model.predict(X_val_s)
val_prob = best_model.predict_proba(X_val_s)

print("Validation metrics")
print("Accuracy          :", round(accuracy_score(y_val, val_pred), 4))
print("Balanced accuracy :", round(balanced_accuracy_score(y_val, val_pred), 4))
print("Macro F1          :", round(f1_score(y_val, val_pred, average='macro'), 4))
print("Weighted F1       :", round(f1_score(y_val, val_pred, average='weighted'), 4))
print()
print(classification_report(y_val, val_pred))

Validation metrics
Accuracy          : 0.9381
Balanced accuracy : 0.9414
Macro F1          : 0.9344
Weighted F1       : 0.938

              precision    recall  f1-score   support

     forward       0.94      0.91      0.93        75
        left       0.87      0.96      0.92        28
       right       0.92      0.89      0.91        38
      update       0.98      1.00      0.99        53

    accuracy                           0.94       194
   macro avg       0.93      0.94      0.93       194
weighted avg       0.94      0.94      0.94       194



In [26]:
# ------------------------------------------------------------------
# HELD-OUT TEST PERFORMANCE (UNSEEN VISUAL PATH)
# ------------------------------------------------------------------

test_pred = best_model.predict(X_test_s)
test_prob = best_model.predict_proba(X_test_s)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "macro_f1": f1_score(y_test, test_pred, average="macro"),
    "weighted_f1": f1_score(y_test, test_pred, average="weighted"),
}

print("Held-out test metrics")
for k, v in test_metrics.items():
    print(f"{k:18s}: {v:.4f}")
print()
print(classification_report(y_test, test_pred))

test_metrics_path = os.path.join(OUTPUT_DIR, "heldout_test_metrics.json")
with open(test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved test metrics to:", test_metrics_path)

Held-out test metrics
accuracy          : 0.9035
balanced_accuracy : 0.8737
macro_f1          : 0.8852
weighted_f1       : 0.9003

              precision    recall  f1-score   support

     forward       0.90      0.94      0.92       127
        left       0.98      0.89      0.93        46
       right       0.83      0.67      0.74        51
      update       0.91      1.00      0.95        87

    accuracy                           0.90       311
   macro avg       0.90      0.87      0.89       311
weighted avg       0.90      0.90      0.90       311

Saved test metrics to: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/heldout_test_metrics.json


In [27]:
# ------------------------------------------------------------------
# CONFUSION MATRIX ON HELD-OUT TEST PATH
# ------------------------------------------------------------------

labels_sorted = sorted(np.unique(np.concatenate([y_test, test_pred])))
cm = confusion_matrix(y_test, test_pred, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("Confusion matrix - held-out VP")
plt.tight_layout()
plt.show()

In [28]:
# ------------------------------------------------------------------
# ATTACH TEST PREDICTIONS TO TEST FEATURE TABLE
# ------------------------------------------------------------------

test_pred_df = test_feat_df.copy()
test_pred_df["pred_label"] = test_pred
test_pred_df["is_correct"] = test_pred_df["label"] == test_pred_df["pred_label"]

class_to_index = {cls: i for i, cls in enumerate(best_model.classes_)}
test_pred_df["pred_confidence"] = [
    float(test_prob[i, class_to_index[test_pred[i]]]) for i in range(len(test_pred))
]

test_pred_csv = os.path.join(OUTPUT_DIR, "heldout_test_predictions.csv")
test_pred_df.to_csv(test_pred_csv, index=False)
print("Saved test predictions to:", test_pred_csv)

display(test_pred_df.head(20))

Saved test predictions to: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_navigator_outputs/heldout_test_predictions.csv


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
0,rep_bed_tv,0,0,forward,update_memory,update,selected_frame_update,False,7.586020e-16,-1.110223e-16,-8.326673e-17,-4.440892e-16,1.000000,2.974163e-17,3.670273e-17,3.502945e-16,1.000000,update,True,0.938792
1,rep_bed_tv,0,3,forward,forward,forward,selected_frame_duplicate_next_target,False,5.682374e-01,-2.588980e-02,-2.374301e-02,2.532331e-01,0.999910,3.370717e-03,1.297198e-02,1.129306e-03,0.904431,forward,True,0.935959
2,rep_bed_tv,1,3,right,right,right,regular_motion,False,4.692756e-01,-3.569933e-02,-2.295490e-02,1.131307e-01,0.999919,3.660933e-03,1.219970e-02,5.302020e-04,0.930474,forward,False,0.699069
3,rep_bed_tv,2,3,forward,forward,forward,regular_motion,False,5.192502e-01,-6.531503e-02,-1.854075e-02,1.032665e-01,-0.999954,-2.414029e-03,9.194846e-03,-8.861523e-04,0.946828,forward,True,0.803807
4,rep_bed_tv,3,3,right,update_memory,update,selected_frame_update,False,1.078586e-15,0.000000e+00,3.053113e-16,-1.332268e-15,1.000000,-1.237869e-17,1.359306e-17,-2.544303e-16,1.000000,update,True,0.938792
5,rep_bed_tv,3,5,right,right,right,selected_frame_duplicate_next_target,False,3.097040e-01,1.989162e-02,-1.775926e-02,7.119942e-02,0.999740,2.373546e-03,2.268937e-02,-6.325140e-04,0.908575,right,True,0.561060
6,rep_bed_tv,4,5,forward,forward,forward,regular_motion,False,3.618185e-01,1.316743e-03,5.482810e-03,7.532176e-02,0.999997,-1.162376e-03,1.912958e-03,-8.819635e-04,0.941193,forward,True,0.713871
7,rep_bed_tv,5,5,right,update_memory,update,selected_frame_update,False,1.076226e-15,-1.110223e-16,-3.053113e-16,-4.440892e-16,-1.000000,5.628806e-17,1.411150e-17,3.401391e-16,1.000000,update,True,0.925583
8,rep_bed_tv,5,7,right,right,right,selected_frame_duplicate_next_target,False,3.532445e-01,3.948934e-02,-1.101012e-02,1.401825e-01,0.999656,1.509478e-03,2.614926e-02,1.182444e-03,0.920854,right,True,0.490760
9,rep_bed_tv,6,7,forward,forward,forward,regular_motion,False,5.228334e-01,-3.343282e-03,-3.470554e-03,7.827487e-02,0.999996,-8.494755e-05,2.655613e-03,-1.089571e-03,0.929403,forward,True,0.788218


In [29]:
# ------------------------------------------------------------------
# VISUAL INSPECTION OF GOOD / BAD PREDICTIONS
# ------------------------------------------------------------------

def show_prediction_row(row: pd.Series, loaded_log: LoadedLog):
    raw_frame_id = int(row["raw_frame_id"])
    target_keyframe = int(row["target_keyframe"])

    observed_rgb = loaded_log.rgb_frames[raw_frame_id]
    key_rgb = loaded_log.rgb_frames[target_keyframe]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(observed_rgb)
    axes[0].set_title(f"Observed RGB | frame {raw_frame_id}")
    axes[0].axis("off")

    axes[1].imshow(key_rgb)
    axes[1].set_title(f"Target key RGB | keyframe {target_keyframe}")
    axes[1].axis("off")

    title = (
        f"true={row['label']} | pred={row['pred_label']} | "
        f"confidence={row['pred_confidence']:.3f} | row_type={row['row_type']}"
    )
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame([row]))


def show_top_examples(df: pd.DataFrame, loaded_log: LoadedLog, correct: bool = True, n: int = 3):
    subset = df[df["is_correct"] == correct].sort_values("pred_confidence", ascending=False).head(n)
    print(f"Showing {'correct' if correct else 'wrong'} examples: {len(subset)}")
    for _, row in subset.iterrows():
        show_prediction_row(row, loaded_log)

In [30]:
# ------------------------------------------------------------------
# SOME GOOD PREDICTIONS
# ------------------------------------------------------------------

show_top_examples(test_pred_df, test_loaded, correct=True, n=4)

Showing correct examples: 4


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
157,rep_bed_tv,110,114,forward,forward,forward,selected_frame_duplicate_next_target,False,0.582215,0.092772,-0.061435,0.396981,0.999937,0.005911,0.009234,-0.002394,0.894066,forward,True,0.998662


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
148,rep_bed_tv,103,105,forward,forward,forward,selected_frame_duplicate_next_target,False,0.778815,0.162098,-0.051801,0.292727,0.999912,0.004742,0.012037,-0.003079,0.904253,forward,True,0.997526


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
158,rep_bed_tv,111,114,forward,forward,forward,regular_motion,False,0.530836,0.024111,-0.052183,0.31875,0.999976,0.006051,0.002959,-0.001296,0.903976,forward,True,0.99665


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
270,rep_bed_tv,192,196,forward,forward,forward,regular_motion,False,0.234551,-0.02418,0.009822,0.389535,-0.999994,0.002257,0.002564,0.000475,0.90526,forward,True,0.994695


In [31]:
# ------------------------------------------------------------------
# SOME BAD PREDICTIONS
# ------------------------------------------------------------------

show_top_examples(test_pred_df, test_loaded, correct=False, n=6)

Showing wrong examples: 6


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
269,rep_bed_tv,191,196,right,right,right,selected_frame_duplicate_next_target,False,0.408414,-0.069785,-0.037724,0.3784,0.999938,0.00494,0.009717,0.002193,0.911891,forward,False,0.989368


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
290,rep_bed_tv,208,212,right,right,right,selected_frame_duplicate_next_target,False,0.025691,-0.005556,-0.0028,0.300139,0.999858,0.000164,0.016872,0.000363,0.917408,forward,False,0.924784


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
140,rep_bed_tv,97,101,right,right,right,regular_motion,False,0.562734,-0.033224,-0.001854,0.297829,0.99989,0.000506,0.014785,0.000992,0.922727,forward,False,0.916231


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
50,rep_bed_tv,34,36,left,left,left,selected_frame_duplicate_next_target,False,0.928945,0.101795,-0.065505,0.127879,-0.999941,-0.009539,0.004744,0.001992,0.922963,forward,False,0.838153


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
11,rep_bed_tv,7,9,right,right,right,selected_frame_duplicate_next_target,False,0.651805,0.16562,0.017906,0.176201,0.999248,-0.002301,0.038654,-0.001944,0.920098,forward,False,0.831032


,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
300,rep_bed_tv,216,219,right,right,right,regular_motion,False,0.018868,0.000895,-0.00027,0.199704,0.999843,-0.00016,0.017699,0.000358,0.926687,forward,False,0.742472


In [32]:
# ------------------------------------------------------------------
# MISCLASSIFICATION SUMMARY
# ------------------------------------------------------------------

mis_df = test_pred_df[~test_pred_df["is_correct"]].copy()
if len(mis_df) > 0:
    display(mis_df[[
        "source_log",
        "raw_frame_id",
        "target_keyframe",
        "label",
        "pred_label",
        "pred_confidence",
        "row_type",
        "rmse",
        "sim_score",
        "tx", "ty", "tz"
    ]].head(30))
else:
    print("No misclassifications in held-out test set.")

,source_log,raw_frame_id,target_keyframe,label,pred_label,pred_confidence,row_type,rmse,sim_score,tx,ty,tz
2,rep_bed_tv,1,3,right,forward,0.699069,regular_motion,0.469276,0.930474,-0.035699,-0.022955,0.113131
11,rep_bed_tv,7,9,right,forward,0.831032,selected_frame_duplicate_next_target,0.651805,0.920098,0.165620,0.017906,0.176201
50,rep_bed_tv,34,36,left,forward,0.838153,selected_frame_duplicate_next_target,0.928945,0.922963,0.101795,-0.065505,0.127879
58,rep_bed_tv,40,42,left,forward,0.472344,regular_motion,0.316932,0.956884,-0.045886,-0.015949,0.073132
75,rep_bed_tv,52,55,left,right,0.710829,selected_frame_duplicate_next_target,1.178389,0.894476,0.000291,-0.008631,-0.006409
80,rep_bed_tv,56,58,left,forward,0.540180,regular_motion,0.485541,0.941285,-0.010058,0.010942,0.109235
83,rep_bed_tv,58,61,forward,left,0.736564,selected_frame_duplicate_next_target,0.246509,0.899859,0.001620,0.015880,0.102591
109,rep_bed_tv,75,79,forward,right,0.359476,selected_frame_duplicate_next_target,1.052202,0.915433,-0.142342,0.013657,0.056627
111,rep_bed_tv,77,79,left,forward,0.373669,regular_motion,0.756474,0.947510,-0.062707,-0.010272,-0.007260
140,rep_bed_tv,97,101,right,forward,0.916231,regular_motion,0.562734,0.922727,-0.033224,-0.001854,0.297829


In [33]:
# ------------------------------------------------------------------
# MANUAL INSPECTION OF A SINGLE TEST ROW
# Edit row_index as needed.
# ------------------------------------------------------------------

row_index = 0
show_prediction_row(test_pred_df.iloc[row_index], test_loaded)

,source_log,raw_frame_id,target_keyframe,user_command,processed_command,label,row_type,bot_lost,rmse,tx,ty,tz,qw,qx,qy,qz,sim_score,pred_label,is_correct,pred_confidence
0,rep_bed_tv,0,0,forward,update_memory,update,selected_frame_update,False,7.586020e-16,-1.110223e-16,-8.326673e-17,-4.440892e-16,1.0,2.974163e-17,3.670273e-17,3.502945e-16,1.0,update,True,0.938792
